# Errors, Files & Modules



## Part 1 — Errors

## The exception hierarchy

Every exception in Python is an instance of a class deriving from `BaseException`. The shape (abbreviated):

```text
   BaseException
     ├─ SystemExit         <- raised by sys.exit() — don't catch this
     ├─ KeyboardInterrupt  <- raised by Ctrl-C — don't catch this
     ├─ GeneratorExit
     └─ Exception          <- THIS is what `except:` should target
          ├─ ArithmeticError → ZeroDivisionError, OverflowError, ...
          ├─ LookupError    → KeyError, IndexError
          ├─ ValueError     → bad input to the right kind
          ├─ TypeError      → operation on wrong type
          ├─ OSError        → file/socket/process errors → FileNotFoundError, PermissionError, ...
          ├─ AttributeError → object doesn't have that attribute
          ├─ RuntimeError   → catch-all for "shouldn't have happened"
          └─ ... (long tail)
```

**The rule**: catch `Exception` or a specific subclass, never `BaseException` and never a bare `except:`. The bare form catches `KeyboardInterrupt` and `SystemExit`, which means Ctrl-C stops working and `sys.exit()` calls get swallowed.

## `try` / `except` / `else` / `finally`

The full shape:

```python
try:
    risky()
except ValueError as e:
    # ran if a ValueError was raised
    handle(e)
except (KeyError, IndexError):
    # multiple types in a tuple — one handler for any of them
    handle_missing()
except Exception as e:
    # catch-all for any other Exception (NOT BaseException)
    log(e)
    raise         # re-raise — pass it up
else:
    # ran if NO exception was raised
    cleanup_after_success()
finally:
    # always runs, exception or not
    close_resources()
```

The `else` clause is the part people miss. It runs only when the `try` block succeeded with no exception. The distinction matters because code in the `try` block that you didn't *expect* to throw might raise an exception that you don't want to be caught by the same `except` handlers.

In [ ]:
def parse_int(s):
    try:
        n = int(s)
    except ValueError as e:
        print(f"  not an integer: {e}")
        return None
    else:
        print(f"  parsed {n}")
        return n
    finally:
        print(f"  done with {s!r}")

print(parse_int("42"))
print("---")
print(parse_int("xyz"))

## Catching exceptions

Three patterns:

```python
# 1. Specific type — preferred
except KeyError as e:
    ...

# 2. Multiple types in one handler — tuple
except (KeyError, IndexError) as e:
    ...

# 3. Catch-all (Exception, not BaseException) — usually last, often re-raises
except Exception as e:
    log(e)
    raise
```

**Never** write a bare `except:`. It catches `KeyboardInterrupt` (your Ctrl-C) and `SystemExit` (your `sys.exit`). If you really need to catch absolutely everything, write `except BaseException:` — but you almost never need to.

In [ ]:
# Multiple types in one handler
def safe_lookup(container, key):
    try:
        return container[key]
    except (KeyError, IndexError, TypeError) as e:
        return f"missing ({type(e).__name__})"

print(safe_lookup({"a": 1}, "b"))      # missing (KeyError)
print(safe_lookup([1, 2, 3], 10))      # missing (IndexError)
print(safe_lookup(None, "x"))          # missing (TypeError)
print(safe_lookup({"a": 1}, "a"))      # 1

## `raise` and exception chaining

`raise` raises an exception. Used three ways:

- **`raise SomeException(msg)`** — raise a new exception.
- **`raise`** alone (inside an `except` block) — re-raise the current exception, preserving the traceback.
- **`raise NewException(...) from original`** — raise a new exception, but link it to the cause via `__cause__`. The default printer shows both: "this caused that."

The `from` form is the right way to translate a low-level exception into a domain-level one without losing the original context.

In [ ]:
# Re-raising — preserves the original traceback
def step():
    try:
        int("xyz")
    except ValueError:
        print("  logging the error...")
        raise

try:
    step()
except ValueError as e:
    print(f"caught at top: {e}")

# Chaining — translate to a domain exception
class ConfigError(Exception):
    pass

def load_port(s):
    try:
        return int(s)
    except ValueError as e:
        raise ConfigError(f"port must be an integer, got {s!r}") from e

try:
    load_port("abc")
except ConfigError as e:
    print(f"\nconfig error: {e}")
    print(f"  caused by:   {e.__cause__}")

## Custom exception classes

Just subclass `Exception`. The convention is to name them with an `Error` suffix and to keep them as flat or as deep a hierarchy as you need for callers to discriminate.

```python
class DataError(Exception): ...
class ValidationError(DataError): ...
class NotFoundError(DataError): ...
```

Now callers can `except DataError:` to catch any of yours, or `except NotFoundError:` for the specific one. Same machinery as the standard library uses.

In [ ]:
class DataError(Exception):
    """Base class for all data-related errors in our app."""

class NotFoundError(DataError):
    def __init__(self, key):
        super().__init__(f"key not found: {key}")
        self.key = key

class ValidationError(DataError):
    pass

def find(key, db):
    if key not in db:
        raise NotFoundError(key)
    return db[key]

try:
    find("missing", {"a": 1})
except DataError as e:        # catches any DataError, including NotFoundError
    print(type(e).__name__, e)
    if isinstance(e, NotFoundError):
        print(f"  bad key was: {e.key}")

## EAFP vs LBYL

Two styles for "is this operation safe?"

- **EAFP** — Easier to Ask Forgiveness than Permission. *Try the operation, catch the exception if it fails.* The Pythonic default.
- **LBYL** — Look Before You Leap. *Check preconditions first, then do the operation.* Common in C, sometimes appropriate.

```python
# EAFP — Pythonic
try:
    value = config["timeout"]
except KeyError:
    value = 30

# LBYL — checks twice in concurrent contexts (TOCTOU race)
if "timeout" in config:
    value = config["timeout"]
else:
    value = 30
```

EAFP is preferred because:

- It's atomic — no race between "check" and "use." In concurrent code, an LBYL check followed by use can have the state change between them.
- It's faster on the success path — exceptions are slow to *raise* but free when they don't fire.
- It maps to how the data actually flows — you describe what you want and handle the cases where it isn't there.

LBYL wins when the precondition check is cheap and the "expected" case is failure (e.g. polling for a file that usually doesn't exist yet).

## The `with` statement and context managers

`with` is Python's resource-management construct — the equivalent of try-with-resources in Java, `using` in C#, `defer` in Go. The shape:

```python
with open("file.txt") as f:
    data = f.read()
# f is closed here, even if an exception was raised inside the block
```

`with` calls the object's `__enter__()` to set up and `__exit__()` to tear down. Use it for anything that needs cleanup: files, network connections, locks, database transactions, temporary directories.

Multiple resources at once — parenthesize since 3.10:

```python
with (open("in.txt") as src,
      open("out.txt", "w") as dst):
    dst.write(src.read())
```

## Writing your own context manager

Two paths:

- **Class form** — define `__enter__` and `__exit__` on a class. `__exit__` returns `True` to suppress an exception, otherwise re-raises.
- **`@contextmanager` decorator** — write a generator function that `yield`s once. Code before the `yield` runs on entry; code after runs on exit. Easier when you don't need exception suppression.

The decorator form is usually cleaner.

In [ ]:
from contextlib import contextmanager
import time

# Class form
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        return False     # don't suppress exceptions

with Timer() as t:
    sum(i * i for i in range(1_000_000))
print(f"elapsed: {t.elapsed:.4f}s")

# @contextmanager — generator form
@contextmanager
def timed(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] {elapsed:.4f}s")

with timed("computation"):
    sum(i * i for i in range(1_000_000))

## `ExceptionGroup` and `except*` — 3.11+

Python 3.11 added **exception groups**, for code that runs many things concurrently and may produce multiple errors at once (typically `asyncio.TaskGroup`).

`ExceptionGroup(message, [list of exceptions])` packages several exceptions into one. The new `except*` syntax handles them group-aware — splits the group, handles matching ones, lets the rest propagate.

You will rarely write `raise ExceptionGroup(...)` by hand. The reason to know it exists: `asyncio.TaskGroup` (notebook 09) uses it to collect failures across concurrent tasks.

In [ ]:
# Manually construct an ExceptionGroup
try:
    raise ExceptionGroup(
        "things broke",
        [ValueError("bad value"), KeyError("missing")],
    )
except* ValueError as eg:
    print(f"caught ValueErrors: {eg.exceptions}")
except* KeyError as eg:
    print(f"caught KeyErrors: {eg.exceptions}")

## Part 2 — Files

## `open()` — the basics

`open(path, mode, encoding=...)` returns a file object. Always use it inside a `with` so the file gets closed.

| Mode | Meaning |
|---|---|
| `"r"` | read text (default) |
| `"w"` | write text — **truncates** existing content |
| `"a"` | append text |
| `"x"` | exclusive create — error if exists |
| `"rb"`, `"wb"`, `"ab"` | binary versions |
| `"r+"`, `"w+"` | read and write |

**Always specify `encoding="utf-8"` for text files.** The default depends on the platform — `cp1252` on Windows, `utf-8` on macOS/Linux — and that platform-dependence is a real source of "works on my machine" bugs. Be explicit.

In [ ]:
# Create a small temp file we'll read back
import tempfile, os
tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8")
tmp.write("line one\nline two\nline three\n")
tmp.close()
path = tmp.name

# Read all at once
with open(path, "r", encoding="utf-8") as f:
    content = f.read()
print(repr(content))

# Read line by line (iteration — memory-efficient)
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        print(line.rstrip())   # rstrip removes the trailing newline

# Read into a list of lines (loads all into memory)
with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()
print(lines)

os.unlink(path)

## Text mode vs binary mode

- **Text mode (`"r"`, `"w"`, default)** — operates on `str`. Lines are split on newlines; line endings are normalized to `\n` regardless of platform. Decoded/encoded using the specified `encoding`.
- **Binary mode (`"rb"`, `"wb"`)** — operates on `bytes`. No newline translation, no encoding. Use for images, executables, anything non-text — and for text files when you need byte-exact control.

You cannot mix them — opening a binary file with `"r"` will eventually fail on something that's not valid UTF-8.

In [ ]:
import tempfile, os

# Write binary
tmp = tempfile.NamedTemporaryFile(suffix=".bin", delete=False)
tmp.write(bytes([0xde, 0xad, 0xbe, 0xef, 0x00, 0xff]))
tmp.close()

# Read binary
with open(tmp.name, "rb") as f:
    raw = f.read()
print(raw, type(raw))
print(raw.hex())

os.unlink(tmp.name)

## `pathlib.Path` — use this, not `os.path`

The modern path API. `Path` is an object that represents a filesystem path, with methods for all the common operations. It replaces the function-based `os.path` interface (`os.path.join`, `os.path.exists`, etc.).

Highlights:

```python
from pathlib import Path

p = Path("data") / "users" / "ganesh.txt"     # / operator joins
p.parent                                       # Path('data/users')
p.name                                         # 'ganesh.txt'
p.stem                                         # 'ganesh'
p.suffix                                       # '.txt'
p.exists()                                     # bool
p.is_file(), p.is_dir()                        # checks
p.read_text(encoding="utf-8")                  # full contents
p.write_text("hello", encoding="utf-8")        # overwrite
p.mkdir(parents=True, exist_ok=True)           # mkdir -p
p.iterdir()                                    # iterate children
p.glob("*.csv")                                # pattern match
p.resolve()                                    # absolute, symlinks resolved
```

In [ ]:
from pathlib import Path
import tempfile

# Work inside a temp dir so we don't pollute the cwd
with tempfile.TemporaryDirectory() as tmpdir:
    base = Path(tmpdir)

    # Compose paths with /
    p = base / "data" / "log.txt"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text("first line\nsecond line\n", encoding="utf-8")

    print(p.exists())            # True
    print(p.name, p.suffix)      # log.txt .txt
    print(p.read_text(encoding="utf-8"))

    # Iterate a directory
    (base / "a.txt").write_text("a")
    (base / "b.txt").write_text("b")
    (base / "sub").mkdir()
    (base / "sub" / "c.txt").write_text("c")

    print("\ntop-level:")
    for child in base.iterdir():
        print(f"  {child.relative_to(base)}  ({'dir' if child.is_dir() else 'file'})")

    # Glob — files matching a pattern
    print("\ntxt files:")
    for f in sorted(base.glob("*.txt")):
        print(f"  {f.name}")

    # Recursive glob — ** matches subdirectories
    print("\nall txt, recursive:")
    for f in sorted(base.rglob("*.txt")):
        print(f"  {f.relative_to(base)}")

## Temporary files and directories

The `tempfile` module gives you safe, auto-cleaning temp files and dirs. Use them in tests, in pipelines that need scratch space, in any "I need a path that nobody else will collide with" situation.

- **`tempfile.TemporaryDirectory()`** — context manager that creates a dir and deletes it on exit.
- **`tempfile.NamedTemporaryFile()`** — context manager that creates a file with a unique name.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "data.txt"
    p.write_text("some scratch data")
    print(f"created: {p}")
    print(f"content: {p.read_text()}")
# After the with block exits, tmpdir and its contents are deleted automatically
print("temp dir cleaned up")

## Part 3 — Modules and packages

## What a module is

A **module** is any `.py` file. When you `import mymodule`, Python loads the file, runs it top to bottom, and gives you a namespace object whose attributes are the names defined in the file.

A **package** is a directory containing an `__init__.py` (or, since 3.3, no `__init__.py` at all in *namespace packages* — but mostly use the explicit form). It groups multiple modules under one importable name.

```text
   myproject/
     __init__.py        <- makes this a package
     core.py            <- a module
     utils/
       __init__.py
       text.py          <- imported as myproject.utils.text
```

## `import`, `from x import y`, `import x as alias`

Four forms, same machinery, different names brought into scope.

| Form | Brings into scope |
|---|---|
| `import math` | `math` (the module). Use as `math.sqrt(2)`. |
| `import math as m` | `m`. Use as `m.sqrt(2)`. |
| `from math import sqrt` | `sqrt`. Use as `sqrt(2)`. |
| `from math import sqrt as sq` | `sq`. Use as `sq(2)`. |
| `from math import *` | everything. **Don't.** Pollutes the namespace, hides where names came from. |

The first form (`import math`) is the safest default — calls are clearly scoped to the module. The `from ... import ...` form is appropriate for names you'll use heavily. The `as` alias is a way out when the imported name would shadow a local one, or when the original name is long.

In [ ]:
import math
from pathlib import Path
import datetime as dt

print(math.pi)
print(Path.cwd())
print(dt.datetime.now())

## `if __name__ == "__main__":` — script vs imported

When Python runs a file, it sets the special variable `__name__`:

- If the file is being **run as a script** (`python myfile.py`), `__name__ == "__main__"`.
- If the file is being **imported** by something else (`import myfile`), `__name__ == "myfile"`.

The idiom:

```python
def main():
    ...

if __name__ == "__main__":
    main()
```

This lets the same file be both an importable module (helper functions get reused) AND a runnable script (the `main()` only fires when you actually run it directly).

You'll see this in almost every standalone Python script. Without it, `import myfile` would execute the entire script as a side effect.

## Where Python looks for modules — `sys.path`

When you `import x`, Python looks for `x.py` (or `x/__init__.py`) in the directories listed in `sys.path`, in order. The search order:

1. The directory of the script being run (or the current directory in an interactive session).
2. The directories listed in the `PYTHONPATH` environment variable.
3. The installation-dependent default paths — your `site-packages/`, where `pip install` puts things.

You can inspect `sys.path` directly. Manipulating it at runtime is occasionally necessary but usually a smell — proper packaging (`pyproject.toml` + `pip install -e .`) is the right answer.

In [ ]:
import sys
for p in sys.path:
    print(p)

## Relative imports — `from . import x`

Inside a package, you can import siblings with **relative imports**: `from . import sibling`, `from .subpkg import thing`. The dots are like `cd` — one dot is "this package," two dots is "the parent package."

Use relative imports for intra-package references. Use absolute imports (`from myproject.core import x`) when crossing package boundaries or in scripts.

```text
   myproject/
     __init__.py
     core.py            <- has: from . import utils
     utils/
       __init__.py      <- has: from .text import slugify
       text.py
```

Relative imports only work inside a package — they fail at the top level. If you see `ImportError: attempted relative import with no known parent package`, you tried a relative import from a script that wasn't packaged.

## `pyproject.toml` and `pip install -e .` — make your project importable

When you have a real project, you want `import myproject` to work from anywhere in the activated venv — not just from the directory that happens to contain `myproject/`.

The modern way: a `pyproject.toml` declaring your project, plus `pip install -e .` to install it in **editable mode** (changes to source are picked up immediately, no reinstall).

Minimal `pyproject.toml`:

```toml
[project]
name = "myproject"
version = "0.1.0"
dependencies = [
    "requests",
    "click",
]

[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"
```

Then from the project root, with your venv activated:

```bash
pip install -e .
```

Now `from myproject.core import x` works from any directory. The `-e` makes it editable — your source files in `src/myproject/` are what gets imported; you don't need to reinstall after every code change.